## Start time of process

In [ ]:
from datetime import datetime
print(datetime.now())

## Initializing files and modules

In [ ]:
%%capture
!pip install --upgrade simpletransformers transformers nltk torch

In [ ]:
import pandas as pd

# Automatically retrieve the needed datasets from the shared Google Drive folder
# If folder is removed, then download the files from here: https://canvas.vu.nl/courses/78637/files/7984201
# Upload these files into the Files section and adjust these lines
hasoc_train = pd.read_csv('https://drive.google.com/uc?export=download&id=173PHI_MhHeiVXUOZsBgdAwoIslBqqkUj')
olid_test = pd.read_csv('https://drive.google.com/uc?export=download&id=1x8_oLad5Kbp2y-g3gm3d-b6SyJd5HRhy')
olid_train_small = pd.read_csv('https://drive.google.com/uc?export=download&id=1YYTEKqrk1ii8qcf3DSyp0JKOr7-v-Smc')

## Code definitions

### Preprocessing

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Preprocess text through lowercasing, tokenization, lemmatization and stopword removal
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    stop_words.update(['user', 'url']) # For Assignment 5, added the removal of 'user' and 'url'.
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

# Apply preprocessing to all datasets
hasoc_train['text'] = hasoc_train['text'].apply(preprocess_text)
olid_test['text'] = olid_test['text'].apply(preprocess_text)
olid_train_small['text'] = olid_train_small['text'].apply(preprocess_text)

### Transformer-based model

In [ ]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs
from sklearn.metrics import classification_report, confusion_matrix
import torch
import os
from datetime import datetime

def train_transformer(train_df, model_type, model_name, desc):
    print(f'\nStart time of training transformer model with {desc}: {datetime.now()}\n')
    # Specify output folder
    os.makedirs(f'transformer_files/outputs', exist_ok=True)
    os.makedirs(f'transformer_files/cache', exist_ok=True)
    os.makedirs(f'transformer_files/best_model', exist_ok=True)
    os.makedirs(f'transformer_files/runs', exist_ok=True)

    # Setting model arguments
    model_args = ClassificationArgs()
    model_args.output_dir = 'transformer_files/outputs'
    model_args.cache_dir = 'transformer_files/cache'
    model_args.best_model_dir = 'transformer_files/best_model'
    model_args.tensorboard_dir = 'transformer_files/runs'
    model_args.num_train_epochs = 4
    model_args.train_batch_size = 16
    model_args.eval_batch_size = 16
    model_args.learning_rate = 2e-5
    model_args.max_seq_length = 128
    model_args.evaluate_during_training = False
    model_args.save_eval_checkpoints = False
    model_args.save_model_every_epoch = True
    model_args.overwrite_output_dir = True
    model_args.no_cache = True # Added due to issues with Google Colab Pro
    model_args.use_multiprocessing = False # Added due to issues with Google Colab Pro
    model_args.use_multiprocessing_for_evaluation = False # Added due to issues with Google Colab Pro

    # Checking if CUDA is available for training
    cuda_available = torch.cuda.is_available()
    print(f'CUDA available: {cuda_available}')

    # Model definition
    model = ClassificationModel(
        model_type,
        model_name,
        use_cuda=cuda_available,
        num_labels=2,
        args=model_args
    )

    # Training the model
    model.train_model(train_df)
    print(f'\nEnd time of training transformer model with {desc}: {datetime.now()}\n')
    return model

def evaluate_transformer(model, test_df, desc):
    print(f'\nStart time of evaluating transformer model with {desc}: {datetime.now()}\n')
    # Evaluating the model
    predictions, _ = model.predict(test_df['text'].tolist())
    print(f'\nEnd time of evaluating transformer model with {desc}: {datetime.now()}\n')
    return classification_report(test_df['labels'], predictions, target_names=['NON', 'OFF'], digits=4), confusion_matrix(test_df['labels'], predictions), predictions

### XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from datetime import datetime

def train_XGBoost(train_df):
    print(f'\nStart time of training model with XGBoost: {datetime.now()}\n')
    # Vectorizing text data
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

    # XGBoost classifier
    xgb_classifier = XGBClassifier(
        max_depth = 6,
        learning_rate = 0.05,
        n_estimators = 300,
        random_state = 42,
        objective='binary:logistic',
    )

    # Model pipeline
    model = Pipeline([
        ('tfidf', vectorizer),
        ('classifier', xgb_classifier)
    ])

    # Training the model
    model.fit(train_df['text'], train_df['labels'])
    print(f'\nEnd time of training model with XGBoost: {datetime.now()}\n')
    return model

def evaluate_XGBoost(model, test_df):
    print(f'\nStart time of evaluating model with XGBoost: {datetime.now()}\n')
    # Evaluating the model
    y_pred = model.predict(test_df['text'])
    print(f'\nEnd time of evaluating model with XGBoost: {datetime.now()}\n')
    return classification_report(test_df['labels'], y_pred, target_names=['NON', 'OFF'], digits=4), confusion_matrix(test_df['labels'], y_pred), y_pred

### Results

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

def print_save_results(results, matrix, model_name):
    os.makedirs('results', exist_ok=True)
    print(f"Results for {model_name}:\n")

    # Classification report
    with open(f'results/{model_name.lower()}_classification_report.txt', 'w') as f:
        f.write(f"Classification report for {model_name}:\n\n")
        f.write(results)
    print(results)
    print(f"\nClassification report saved.\n\n")

    # Confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16})
    plt.ylabel('True label', fontsize=16)
    plt.xlabel('Predicted label', fontsize=16)
    plt.xticks(ticks=[0.5, 1.5], labels=['NON', 'OFF'], fontsize=16)
    plt.yticks(ticks=[0.5, 1.5], labels=['NON', 'OFF'], fontsize=16)
    plt.savefig(f'results/{model_name.lower()}_confusion_matrix.png')
    plt.show()
    print('\nConfusion matrix saved.\n\n')
    plt.close()

### Error analysis

In [ ]:
import numpy as np

def error_analysis(model, test_data, y_pred, model_name, model_type='transformer'):
    # Defining mini function to write to file and print.
    with open(f'results/{model_name.lower()}_error_analysis.txt', 'w') as f:
        def write_and_print(message):
            f.write(message + '\n')
            print(message)

        write_and_print(f"Error Analysis for {model_name}:")
        y_test = test_data['labels']

        # Feature importance analysis for XGBoost model
        if model_type == 'xgboost':
            # Access the TF-IDF vectorizer and classifier
            tfidf_vectorizer = model.named_steps['tfidf']
            classifier = model.named_steps['classifier']
            feature_names = tfidf_vectorizer.get_feature_names_out()
            feature_importance = classifier.feature_importances_

            # Ensure feature_importance matches the number of feature_names
            if len(feature_importance) > len(feature_names):
                feature_importance = feature_importance[:len(feature_names)]
            sorted_idx = np.argsort(feature_importance)[::-1]
            top_features = [feature_names[i] for i in sorted_idx[:10]]

            write_and_print("\nTop 10 important features:")
            write_and_print(", ".join(top_features))

        else: # Transformer models do not have feature importance analysis
            write_and_print("\nFeature importance analysis not available for transformer models.")

        # Anaylsis of false positives/negatives
        # true_positives = ((y_pred == 1) & (y_test == 1))
        # true_negatives = ((y_pred == 0) & (y_test == 0))
        false_positives = ((y_pred == 1) & (y_test == 0))
        false_negatives = ((y_pred == 0) & (y_test == 1))

        # write_and_print(f"\n- True Positives: {true_positives.sum()}")
        # write_and_print(f"\n- True Negatives: {true_negatives.sum()}")
        write_and_print(f"\n- False Positives: {false_positives.sum()}")
        write_and_print(f"\n- False Negatives: {false_negatives.sum()}")

        # Example analysis
        write_and_print("\nExample Analysis:")
        def print_examples(condition, label, num_examples=5):
            indices = np.where(condition)[0]
            write_and_print(f"\n{label} Examples:")
            for i in range(min(num_examples, len(indices))):
                idx = indices[i]
                text = test_data['text'].iloc[idx]
                true_label = y_test.iloc[idx]
                pred_label = y_pred[idx]
                write_and_print(f"\nText: {text}")
                write_and_print(f"- True Label: {'Offensive' if true_label == 1 else 'Non-offensive'}")
                write_and_print(f"- Predicted Label: {'Offensive' if pred_label == 1 else 'Non-offensive'}")

        # print_examples(true_positives, "True Positive")
        # print_examples(true_negatives, "True Negative")
        print_examples(false_positives, "False Positive")
        print_examples(false_negatives, "False Negative")

    print(f"\nError analysis saved.")

## In-domain experiments

### In-domain HateBERT

In [ ]:
hatebert_model_in = train_transformer(olid_train_small, 'bert', 'GroNLP/hateBERT', 'in-domain_HateBERT')
hatebert_report_in, hatebert_confusion_matrix_in, hatebert_predictions_in = evaluate_transformer(hatebert_model_in, olid_test, 'in-domain_HateBERT')

print_save_results(hatebert_report_in, hatebert_confusion_matrix_in, '4_in-domain_HateBERT')
error_analysis(hatebert_model_in, olid_test, hatebert_predictions_in, '4_in-domain_HateBERT', model_type='bert')

### In-domain roBERTa

In [ ]:
roberta_model_in = train_transformer(olid_train_small, 'roberta', 'FacebookAI/roberta-base', 'in-domain_roBERTa')
roberta_report_in, roberta_confusion_matrix_in, roberta_predictions_in = evaluate_transformer(roberta_model_in, olid_test, 'in-domain_roBERTa')

print_save_results(roberta_report_in, roberta_confusion_matrix_in, '4_in-domain_roBERTa')
error_analysis(roberta_model_in, olid_test, roberta_predictions_in, '4_in-domain_roBERTa', model_type='roberta')

### In-domain XGBoost

In [ ]:
xgboost_model_in = train_XGBoost(olid_train_small)
xgboost_report_in, xgboost_confusion_matrix_in, xgboost_predictions_in = evaluate_XGBoost(xgboost_model_in, olid_test)

print_save_results(xgboost_report_in, xgboost_confusion_matrix_in, '4_in-domain_XGBoost')
error_analysis(xgboost_model_in, olid_test, xgboost_predictions_in, '4_in-domain_XGBoost', model_type='xgboost')

## Cross-domain experiments

### Cross-domain HateBERT

In [ ]:
hatebert_model_cross = train_transformer(hasoc_train, 'bert', 'GroNLP/hateBERT', 'cross-domain_HateBERT')
hatebert_report_cross, hatebert_confusion_matrix_cross, hatebert_predictions_cross = evaluate_transformer(hatebert_model_cross, olid_test, 'cross-domain_HateBERT')

print_save_results(hatebert_report_cross, hatebert_confusion_matrix_cross, '4_cross-domain_HateBERT')
error_analysis(hatebert_model_cross, olid_test, hatebert_predictions_cross, '4_cross-domain_HateBERT', model_type='bert')

### Cross-domain roBERTa

In [ ]:
roberta_model_cross = train_transformer(hasoc_train, 'roberta', 'FacebookAI/roberta-base', 'cross-domain_roBERTa')
roberta_report_cross, roberta_confusion_matrix_cross, roberta_predictions_cross = evaluate_transformer(roberta_model_cross, olid_test, 'cross-domain_roBERTa')

print_save_results(roberta_report_cross, roberta_confusion_matrix_cross, '4_cross-domain_roBERTa')
error_analysis(roberta_model_cross, olid_test, roberta_predictions_cross, '4_cross-domain_roBERTa', model_type='roberta')

### Cross-domain XGBoost

In [ ]:
xgboost_model_cross = train_XGBoost(hasoc_train)
xgboost_report_cross, xgboost_confusion_matrix_cross, xgboost_predictions_cross = evaluate_XGBoost(xgboost_model_cross, olid_test)

print_save_results(xgboost_report_cross, xgboost_confusion_matrix_cross, '4_cross-domain_XGBoost')
error_analysis(xgboost_model_cross, olid_test, xgboost_predictions_cross, '4_cross-domain_XGBoost', model_type='xgboost')

# Ensemble Methods
Newly developed code for Assignment 5, with new results and error analysis function which are adapted from Assignment 4.

## Code definitions

### Hard Majority Voting

In [ ]:
from sklearn.metrics import accuracy_score

def hard_majority_voting(hatebert_predictions, roberta_predictions, xgboost_predictions, domain):
  # Combine predictions
  ensemble_predictions = []
  for i in range(len(hatebert_predictions)):
    votes = [hatebert_predictions[i], roberta_predictions[i], xgboost_predictions[i]]
    majority_vote = max(set(votes), key=votes.count)
    ensemble_predictions.append(majority_vote)

  # Evaluate ensemble
  accuracy = accuracy_score(olid_test['labels'], ensemble_predictions)
  return accuracy, ensemble_predictions

### Soft Majority Voting

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
import torch
import torch.nn.functional as F

def get_probabilities(model, input, model_type, train_data):
    if model_type == 'transformer':
        print("Transformer model does not have predict_proba. Using logits and applying softmax for probabilities.")

        # Predict raw logits from transformer model
        predictions, _ = model.predict(input.tolist())

        # Convert logits to tensor
        logits = torch.tensor(predictions, dtype=torch.float32)

        # Convert logits to two classes
        if logits.dim() == 1:
            logits = torch.stack([logits, 1 - logits], dim=1)

        # Apply softmax along the last dimension (for class probabilities)
        probs = F.softmax(logits, dim=-1).numpy()
        return probs

    else:
        # If not transformer, use regular model and calibration
        print("Using calibration for non-transformer model.")
        calibrated_model = CalibratedClassifierCV(estimator=model, method='sigmoid', cv=5)
        calibrated_model.fit(train_data['text'], train_data['labels'])
        return calibrated_model.predict_proba(input)

def soft_majority_voting(probs_list, true_labels, domain):
    # Convert all probabilities to numpy arrays to ensure consistency
    probs_list = [np.array(probs) for probs in probs_list]

    # Ensure that all models have the same number of classes and samples
    n_samples = probs_list[0].shape[0]
    n_classes = probs_list[0].shape[1]

    for probs in probs_list:
        assert probs.shape == (n_samples, n_classes), f"Shape mismatch: Expected {(n_samples, n_classes)}, got {probs.shape}"

    # Average the probabilities across the models (soft voting)
    ensemble_probs = np.mean(probs_list, axis=0)

    # Get predicted labels from ensemble probabilities
    ensemble_predictions = np.argmax(ensemble_probs, axis=1)

    # Evaluate ensemble accuracy
    accuracy = accuracy_score(true_labels, ensemble_predictions)
    return accuracy, ensemble_predictions

### Stacking Ensemble

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

def stacking_ensemble(models, train_data, test_data, domain, add_features=True, k=5):
    # Get Out-of-Fold Predictions
    def get_oof_predictions(model, X, y, k):
        kf = KFold(n_splits=k, shuffle=True, random_state=42)
        oof_predictions = np.zeros((len(y), 2))  # binary classification, 2 classes
        for train_idx, val_idx in kf.split(X):
            if hasattr(model, 'predict'):
                prediction = model.predict(X.iloc[val_idx].tolist())
                logits = prediction[0] if isinstance(prediction, tuple) else prediction
                logits = torch.tensor(logits, dtype=torch.float32)
                probs = F.softmax(logits, dim=-1).numpy()  # Apply softmax to get probabilities
                if probs.ndim == 1:
                    oof_predictions[val_idx, 1] = probs
                    oof_predictions[val_idx, 0] = 1 - probs
                else:
                    oof_predictions[val_idx] = probs
            else:
                model.fit(X.iloc[train_idx], y.iloc[train_idx])
                probs = model.predict_proba(X.iloc[val_idx])
                if probs.shape[1] == 1:
                    oof_predictions[val_idx, 1] = probs.ravel()
                    oof_predictions[val_idx, 0] = 1 - probs.ravel()
                else:
                    oof_predictions[val_idx] = probs
        return oof_predictions

    # Get OOF predictions for training data
    oof_predictions = []
    for model in models:
        oof_preds = get_oof_predictions(model, train_data['text'], train_data['labels'], k)
        oof_predictions.append(oof_preds)

    # Create Feature Matrix for Meta-learner
    stacking_features = np.column_stack(oof_predictions)

    # Function to create additional features
    def create_additional_features(data, is_train=True):
        if is_train:
            # Create and fit TF-IDF and Word2Vec only during training
            global tfidf_vectorizer, w2v_model
            tfidf_vectorizer = TfidfVectorizer(max_features=1000)
            tfidf_features = tfidf_vectorizer.fit_transform(data['text']).toarray()

            sentences = [simple_preprocess(text) for text in data['text']]
            w2v_model = Word2Vec(sentences, vector_size=200, window=5, min_count=1, workers=4, sg=1)
            w2v_features = np.array([np.mean([w2v_model.wv[word] for word in simple_preprocess(text) if word in w2v_model.wv] or [np.zeros(200)], axis=0) for text in data['text']])
        else:
            # Use pre-fitted TF-IDF and Word2Vec for test data
            tfidf_features = tfidf_vectorizer.transform(data['text']).toarray()
            w2v_features = np.array([np.mean([w2v_model.wv[word] for word in simple_preprocess(text) if word in w2v_model.wv] or [np.zeros(200)], axis=0) for text in data['text']])

        return np.hstack((tfidf_features, w2v_features))

    # Add Additional Features (Optional)
    if add_features:
        additional_features = create_additional_features(train_data, is_train=True)
        stacking_features = np.hstack((stacking_features, additional_features))

    # Train Meta-learner
    meta_learner = LogisticRegression()
    meta_learner.fit(stacking_features, train_data['labels'])

    # Make Predictions on Test Data
    test_oof_predictions = []
    for model in models:
        if hasattr(model, 'predict'):
            prediction = model.predict(test_data['text'].tolist())
            logits = prediction[0] if isinstance(prediction, tuple) else prediction
            logits = torch.tensor(logits, dtype=torch.float32)
            probs = F.softmax(logits, dim=-1).numpy()  # Apply softmax to logits
            if probs.ndim == 1:
                probs = np.column_stack((1 - probs, probs))
            test_oof_predictions.append(probs)
        else:
            probs = model.predict_proba(test_data['text'])
            if probs.shape[1] == 1:
                probs = np.column_stack((1 - probs.ravel(), probs.ravel()))
            test_oof_predictions.append(probs)

    # Stacking test features
    test_stacking_features = np.column_stack(test_oof_predictions)
    if add_features:
        test_additional_features = create_additional_features(test_data, is_train=False)
        test_stacking_features = np.hstack((test_stacking_features, test_additional_features))

    ensemble_predictions = meta_learner.predict(test_stacking_features)
    accuracy = accuracy_score(test_data['labels'], ensemble_predictions)

    return accuracy, ensemble_predictions

### Results

In [ ]:
# [DONE] Copy results function of Assignment 4 and adjust if needed for ensemble methods

import matplotlib.pyplot as plt
import seaborn as sns

def ensemble_print_save_results(y_true, y_pred, ensemble_name):
    print(f"Results for {ensemble_name}:\n")

    # Classification report
    report = classification_report(y_true, y_pred, target_names=['NON', 'OFF'], digits=4)
    with open(f'results/{ensemble_name.lower()}_classification_report.txt', 'w') as f:
        f.write(f"Classification report for {ensemble_name}:\n\n")
        f.write(report)
    print(report)
    print(f"\nClassification report saved.\n\n")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16})
    plt.ylabel('True label', fontsize=16)
    plt.xlabel('Predicted label', fontsize=16)
    plt.xticks(ticks=[0.5, 1.5], labels=['NON', 'OFF'], fontsize=16)
    plt.yticks(ticks=[0.5, 1.5], labels=['NON', 'OFF'], fontsize=16)
    plt.savefig(f'results/{ensemble_name.lower()}_confusion_matrix.png')
    plt.show()
    print('\nConfusion matrix saved.\n\n')
    plt.close()

### Error analysis

In [ ]:
# [DONE] Copy results function of Assignment 4 and adjust if needed for ensemble methods
# [DONE] Remove true positives and true negatives as Assignment 5 does not focus on that

import numpy as np

def ensemble_error_analysis(test_data, y_pred, ensemble_name):
    # Defining mini function to write to file and print.
    with open(f'results/{ensemble_name.lower()}_error_analysis.txt', 'w') as f:
        def write_and_print(message):
            f.write(message + '\n')
            print(message)

        write_and_print(f"Error Analysis for {ensemble_name}:")
        y_test = test_data['labels']
        y_pred = np.array(y_pred) if isinstance(y_pred, list) else y_pred # Adjust list to np.array if needed

        # Anaylsis of false positives/negatives
        false_positives = ((y_pred == 1) & (y_test == 0))
        false_negatives = ((y_pred == 0) & (y_test == 1))
        write_and_print(f"\n- False Positives: {false_positives.sum()}")
        write_and_print(f"\n- False Negatives: {false_negatives.sum()}")

        # Example analysis
        write_and_print("\nExample Analysis:")
        def print_examples(condition, label, num_examples=5):
            indices = np.where(condition)[0]
            write_and_print(f"\n{label} Examples:")
            for i in range(min(num_examples, len(indices))):
                idx = indices[i]
                text = test_data['text'].iloc[idx]
                true_label = y_test.iloc[idx]
                pred_label = y_pred[idx]
                write_and_print(f"\nText: {text}")
                write_and_print(f"- True Label: {'Offensive' if true_label == 1 else 'Non-offensive'}")
                write_and_print(f"- Predicted Label: {'Offensive' if pred_label == 1 else 'Non-offensive'}")

        print_examples(false_positives, "False Positive")
        print_examples(false_negatives, "False Negative")

    print(f"\nError analysis saved.")

### Comparison

In [ ]:
# [DONE] Implement the printing of all accuracies
# [DONE] Implement checking the performance of ensemble compared to individuals
# [DONE]    Optional: implement McNemar's test for significance between best ensemble / individual models?
# [DONE] Implement Pearson Correlations Coefficents between all in-models
# [DONE]    Optional: into a heatmap?
# [DONE]    Optional: implement interpretation of each correlation? R < 0.5 and < 0.8 for low, moderate, high?

from scipy.stats import pearsonr
from statsmodels.stats.contingency_tables import mcnemar

def ensemble_comparison(models_dict, domain):
    def write_and_print(message):
        with open(f'results/5_comparison_{domain.lower()}.txt', 'a') as f:
            f.write(message + '\n')
        print(message)

    write_and_print(f"Analysis for {domain} models:")

    # Compare performance
    write_and_print("\nModel Performances:")
    accuracies = {}
    for name, preds in models_dict.items():
        accuracy = accuracy_score(olid_test['labels'], preds)
        accuracies[name] = accuracy
        write_and_print(f"- {name}: Accuracy = {accuracy:.4f}")

    # Check if ensemble models improved over individual models
    ensemble_methods = ['Ensemble', 'MV', 'SE', 'Stacking']
    individual_models = [model for model in models_dict.keys() if not any(method in model for method in ensemble_methods)]
    ensemble_models = [model for model in models_dict.keys() if any(method in model for method in ensemble_methods)]
    best_individual = max((accuracies[model], model) for model in individual_models)
    best_ensemble = max((accuracies[model], model) for model in ensemble_models)

    # Check if the ensemble methods improved performance over the best individual model
    if best_ensemble[0] > best_individual[0]:
        write_and_print(f"\nEnsemble approach improved performance.")
    else:
        write_and_print(f"\nEnsemble approach did not improve performance.")
    write_and_print(f"- Best individual ({best_individual[1]}): {best_individual[0]:.4f}")
    write_and_print(f"- Best ensemble ({best_ensemble[1]}): {best_ensemble[0]:.4f}")

    # Optional: implement McNemar's test for significance between best ensemble / individual models?
    def create_contingency_table(model_1_preds, model_2_preds, true_labels):
      table = [[0, 0], [0, 0]]
      for model_1, model_2, true in zip(model_1_preds, model_2_preds, true_labels):
          if model_1 == true:
              if model_2 == true:
                  table[1][1] += 1  # Both correct
              else:
                  table[1][0] += 1  # Model1 correct, Model2 incorrect
          else:
              if model_2 == true:
                  table[0][1] += 1  # Model1 incorrect, Model2 correct
              else:
                  table[0][0] += 1  # Both incorrect
      return table

    best_individual_preds = models_dict[best_individual[1]]
    best_ensemble_preds = models_dict[best_ensemble[1]]
    table = create_contingency_table(best_individual_preds, best_ensemble_preds, olid_test['labels'])
    result = mcnemar(table, exact=False, correction=True)
    write_and_print(f"McNemar's test result:")
    write_and_print(f"- statistic: {result.statistic:.4f}, p-value: {result.pvalue:.4f}")

    # Calculate Pearson correlation coefficients
    model_names = list(models_dict.keys())
    correlations = np.zeros((len(model_names), len(model_names)))
    for model_1, (name_1, preds_1) in enumerate(models_dict.items()):
        for model_2, (name_2, preds_2) in enumerate(models_dict.items()):
            if model_1 <= model_2:  # We only need to calculate the upper triangle since it is symmetric
                corr, _ = pearsonr(preds_1, preds_2)
                correlations[model_1, model_2] = corr
                correlations[model_2, model_1] = corr
    correlation_df = pd.DataFrame(correlations, index=model_names, columns=model_names)
    print(f"\nPearson Correlation Coefficients:\n{correlation_df}\n")

    # Optional: into a heatmap?
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, annot_kws={"size": 16})
    plt.title(f'Correlation Heatmap of Model Predictions ({domain})', fontsize=16)
    plt.tight_layout()
    plt.savefig(f'results/5_correlations_{domain.lower()}.png')
    plt.show()
    write_and_print(f"\nPearson Correlation Coefficients heatmap saved.")

    # Optional: implement interpretation of each correlation? R < 0.5 and < 0.8 for low, moderate, high?
    write_and_print("\nInterpretation of Correlations:")
    for model_1 in range(len(model_names)):
        for model_2 in range(model_1+1, len(model_names)):
            corr = correlations[model_1, model_2]
            if abs(corr) > 0.8: # abs() makes sure that the if-statement checks positive and negative directions
                write_and_print(f"- {model_names[model_1]} and {model_names[model_2]} have highly correlated predictions (r = {corr:.2f})")
            elif abs(corr) < 0.5:
                write_and_print(f"- {model_names[model_1]} and {model_names[model_2]} have relatively uncorrelated predictions (r = {corr:.2f})")
            else:
                write_and_print(f"- {model_names[model_1]} and {model_names[model_2]} have moderately correlated predictions (r = {corr:.2f})")
    write_and_print("Positive = same predictions, negative = opposite predictions.")

    print(f"\nComparison saved.")

## In-domain experiments

### In-domain Hard Majority Voting

In [ ]:
in_domain_hard_accuracy, in_domain_hard_predictions = hard_majority_voting(hatebert_predictions_in, roberta_predictions_in, xgboost_predictions_in, 'in-domain')

ensemble_print_save_results(olid_test['labels'], in_domain_hard_predictions, '5_in-domain_hard-majority-voting')
ensemble_error_analysis(olid_test, in_domain_hard_predictions, '5_in-domain_hard-majority-voting')

### In-domain Soft Majority Voting

In [ ]:
hatebert_probs_in = get_probabilities(hatebert_model_in, olid_test['text'], 'transformer', olid_test )
roberta_probs_in = get_probabilities(roberta_model_in, olid_test['text'], 'transformer', olid_test)
xgboost_probs_in = get_probabilities(xgboost_model_in, olid_test['text'], 'other', olid_test)
in_domain_soft_accuracy, in_domain_soft_predictions = soft_majority_voting([hatebert_probs_in, roberta_probs_in, xgboost_probs_in], olid_test['labels'], 'in-domain')

ensemble_print_save_results(olid_test['labels'], in_domain_soft_predictions, '5_in-domain_soft-majority-voting')
ensemble_error_analysis(olid_test, in_domain_soft_predictions, '5_in-domain_soft-majority-voting')

### In-domain Stacking Ensemble

In [ ]:
models_in = [hatebert_model_in, roberta_model_in, xgboost_model_in]
in_domain_stacking_accuracy, in_domain_stacking_predictions = stacking_ensemble(models_in, olid_train_small, olid_test, 'in-domain', add_features=True)

ensemble_print_save_results(olid_test['labels'], in_domain_stacking_predictions, '5_in-domain_stacking-ensemble')
ensemble_error_analysis(olid_test, in_domain_stacking_predictions, '5_in-domain_stacking-ensemble')

### In-domain Comparisons (all 6)

In [ ]:
in_domain_models = {
    'HateBERT': hatebert_predictions_in,
    'RoBERTa': roberta_predictions_in,
    'XGBoost': xgboost_predictions_in,
    'Hard MV': in_domain_hard_predictions,
    'Soft MV': in_domain_soft_predictions,
    'Stacking': in_domain_stacking_predictions
}

ensemble_comparison(in_domain_models, 'in-domain')

## Cross-domain experiments

### Cross-domain Hard Majority Voting

In [ ]:
cross_domain_hard_accuracy, cross_domain_hard_predictions = hard_majority_voting(hatebert_predictions_cross, roberta_predictions_cross, xgboost_predictions_cross, 'cross-domain')

ensemble_print_save_results(olid_test['labels'], cross_domain_hard_predictions, '5_cross-domain_hard-majority-voting')
ensemble_error_analysis(olid_test, cross_domain_hard_predictions, '5_cross-domain_hard-majority-voting')

### Cross-domain Soft Majority Voting


In [ ]:
hatebert_probs_cross = get_probabilities(hatebert_model_cross, olid_test['text'], 'transformer', olid_test )
roberta_probs_cross = get_probabilities(roberta_model_cross, olid_test['text'], 'transformer', olid_test)
xgboost_probs_cross = get_probabilities(xgboost_model_cross, olid_test['text'], 'other', olid_test)
cross_domain_soft_accuracy, cross_domain_soft_predictions = soft_majority_voting([hatebert_probs_cross, roberta_probs_cross, xgboost_probs_cross], olid_test['labels'], 'cross-domain')

ensemble_print_save_results(olid_test['labels'], cross_domain_soft_predictions, '5_cross-domain_soft-majority-voting')
ensemble_error_analysis(olid_test, cross_domain_soft_predictions, '5_cross-domain_soft-majority-voting')

### Cross-domain Stacking Ensemble

In [ ]:
models_cross = [hatebert_model_cross, roberta_model_cross, xgboost_model_cross]
cross_domain_stacking_accuracy, cross_domain_stacking_predictions = stacking_ensemble(models_cross, hasoc_train, olid_test, 'cross-domain', add_features=True)

ensemble_print_save_results(olid_test['labels'], cross_domain_stacking_predictions, '5_cross-domain_stacking-ensemble')
ensemble_error_analysis(olid_test, cross_domain_stacking_predictions, '5_cross-domain_stacking-ensemble')

### Cross-domain Comparisons (all 6)

In [ ]:
cross_domain_models = {
    'HateBERT': hatebert_predictions_cross,
    'RoBERTa': roberta_predictions_cross,
    'XGBoost': xgboost_predictions_cross,
    'Hard MV': cross_domain_hard_predictions,
    'Soft MV': cross_domain_soft_predictions,
    'Stacking': cross_domain_stacking_predictions
}

ensemble_comparison(cross_domain_models, 'cross-domain')

## Compressing and downloading all files

In [ ]:
!rm -rf sample_data

In [ ]:
!zip -r /content/A5_results.zip /content/results
from google.colab import files
files.download("/content/A5_results.zip")

## End time of process

In [ ]:
from datetime import datetime
print(datetime.now())